# Bài tập Thực hành: Phân loại Âm thanh - Speech Commands Dataset (MFCCs + SVM)

Trong bài tập này, các bạn sẽ thực hành xây dựng và đánh giá quy trình Phân loại Lệnh Giọng nói (Speech Commands) sử dụng phương pháp Học máy truyền thống. Quy trình bao gồm các bước cốt lõi:

1. **Tải & Lọc Dữ liệu Âm thanh**: Tải tự động bộ dữ liệu `Speech Commands Dataset` bằng `torchaudio.datasets` và lọc ra tập con gồm 10 từ khóa cơ bản để tối ưu thời gian huấn luyện.
2. **Tiền xử lý Độ dài Tín hiệu (Padding/Truncating)**: Do các file âm thanh có độ dài không đồng đều, bạn sẽ sử dụng `torch.nn.functional.pad` để chuẩn hóa tất cả các mảng `waveform` về đúng kích thước cố định `(1, 16000)` (tương đương 1 giây).
3. **Trích xuất Đặc trưng MFCCs**: Khởi tạo biến đổi `torchaudio.transforms.MFCC` để trích xuất đặc trưng Cepstral từ tín hiệu âm thanh.
4. **Chuyển đổi Không gian Chiều (Mean Pooling)**: Xử lý tensor MFCCs 2D đầu ra thành vector đặc trưng 1D bằng cách tính trung bình dọc theo trục thời gian (`dim=-1`), nhằm đáp ứng yêu cầu đầu vào của mô hình SVM.
5. **Huấn luyện Mô hình SVM**: Sử dụng thư viện `sklearn.svm.SVC` để huấn luyện mô hình phân loại đa lớp trên tập đặc trưng MFCCs 1D.
6. **Đánh giá & Trực quan hóa**: Sử dụng `sklearn.metrics` để tính toán và hiển thị **Classification Report**, đồng thời vẽ ma trận nhầm lẫn để phân tích độ chính xác của mô hình.
7. **Câu hỏi thảo luận**: Phân tích tầm quan trọng của việc chuẩn hóa độ dài tín hiệu và cơ chế giải tương quan của biến đổi DCT trong thuật toán MFCCs.

### 1. Import các thư viện cần thiết

In [ ]:
import os
import torch
import torchaudio
import torchaudio.transforms as T
import numpy as np
from torch.nn.functional import pad
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

# Fix random seed để kết quả ổn định
torch.manual_seed(42)
np.random.seed(42)

### 2. Tải và Lọc bộ dữ liệu Speech Commands
Bộ dữ liệu Speech Commands chứa 35 từ khác nhau. Để mô hình SVM huấn luyện nhanh và hiệu quả trong khuôn khổ bài thực hành, chúng ta sẽ lọc ra **10 từ khóa cơ bản** (ví dụ: *yes, no, up, down, left, right, on, off, stop, go*).

In [ ]:
# Khai báo danh sách các nhãn (labels) cần phân loại
TARGET_WORDS = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']
word_to_idx = {word: i for i, word in enumerate(TARGET_WORDS)}

print("Đang tải Speech Commands Dataset...")
DATA_ROOT = os.path.join(os.getcwd(), "speech_commands_data")
train_dataset = torchaudio.datasets.SPEECHCOMMANDS(
    root=DATA_ROOT,
    download=True,
    subset="training",
)
test_dataset = torchaudio.datasets.SPEECHCOMMANDS(
    root=DATA_ROOT,
    download=False,
    subset="testing",
)


def filter_dataset(dataset):
    filtered_data = []
    for waveform, sample_rate, label, *_ in dataset:
        if label in TARGET_WORDS:
            filtered_data.append((waveform, sample_rate, word_to_idx[label]))
    return filtered_data


train_data = filter_dataset(train_dataset)
test_data = filter_dataset(test_dataset)

print(f"Train samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

### 3. Khởi tạo bộ trích xuất MFCCs

In [ ]:
# Khởi tạo MFCC extractor
sample_rate = 16000
mfcc_transform = T.MFCC(
    sample_rate=sample_rate,
    n_mfcc=13,
    melkwargs={
        "n_fft": 400,
        "hop_length": 160,
        "n_mels": 40,
    },
)

print("MFCC transform đã được khởi tạo.")

### 4. Tiền xử lý (Padding/Truncating) và Trích xuất Đặc trưng
**Vấn đề:** Các file âm thanh trong Speech Commands có độ dài tối đa là 1 giây (16000 samples), nhưng có nhiều file bị cắt ngắn hơn (ví dụ: 14000, 15000 samples).

**Giải pháp:** Cần chuẩn hóa mảng `waveform` về đúng kích thước cố định `(1, 16000)` trước khi đưa qua hàm tính MFCC. Sau đó áp dụng `.mean(dim=-1)` để chuyển tensor 2D thành vector 1D.

In [ ]:
# TODO 3: Viết hàm tiền xử lý và trích xuất MFCC

def preprocess_and_extract_mfcc(dataset, transform, max_length=16000):
    X = []
    y = []

    for waveform, sr, label in dataset:
        if sr != sample_rate:
            waveform = torchaudio.functional.resample(
                waveform,
                orig_freq=sr,
                new_freq=sample_rate,
            )

        if waveform.size(0) > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if waveform.size(1) < max_length:
            waveform = pad(waveform, (0, max_length - waveform.size(1)))
        else:
            waveform = waveform[:, :max_length]

        mfcc = transform(waveform)
        feature_vector = mfcc.mean(dim=-1).squeeze(0)

        X.append(feature_vector.numpy())
        y.append(label)

    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.int64)


X_train, y_train = preprocess_and_extract_mfcc(train_data, mfcc_transform)
X_test, y_test = preprocess_and_extract_mfcc(test_data, mfcc_transform)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")

### 5. Huấn luyện SVM

In [ ]:
# TODO 4: Khởi tạo và huấn luyện mô hình SVM
svm_model = SVC(
    kernel="rbf",
    C=10.0,
    gamma="scale",
    random_state=42,
)

print("Đang huấn luyện SVM...")
svm_model.fit(X_train, y_train)
print("Huấn luyện SVM hoàn tất.")

### 6. Đánh giá Mô hình

In [ ]:
# TODO 5: Đánh giá độ chính xác trên tập Test
from sklearn.metrics import confusion_matrix
import seaborn as sns

predictions = svm_model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Test accuracy: {accuracy * 100:.2f}%")
print(
    classification_report(
        y_test,
        predictions,
        labels=list(range(len(TARGET_WORDS))),
        target_names=TARGET_WORDS,
        zero_division=0,
    )
)

confusion = confusion_matrix(
    y_test,
    predictions,
    labels=list(range(len(TARGET_WORDS))),
)

plt.figure(figsize=(10, 8))
sns.heatmap(
    confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=TARGET_WORDS,
    yticklabels=TARGET_WORDS,
)
plt.title("Confusion Matrix - MFCC + SVM")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()
plt.show()

### 7. Thảo luận
**Câu hỏi 1:** Tại sao với Speech Commands Dataset, bước kiểm tra và chuẩn hóa độ dài tín hiệu (Padding/Truncating) lại bắt buộc phải thực hiện trước khi trích xuất đặc trưng? Hiện tượng gì sẽ xảy ra đối với đầu vào của SVM nếu ta bỏ qua bước này?

*Trả lời của sinh viên:*

Các file âm thanh trong Speech Commands không nhất thiết có cùng số lượng mẫu. Nếu không chuẩn hóa độ dài, tensor waveform và tensor MFCC sẽ có số frame thời gian khác nhau. Khi đó không thể ghép các mẫu thành một ma trận đặc trưng thống nhất để đưa vào SVM, vì SVM yêu cầu tất cả mẫu có cùng số chiều đầu vào.

Padding bằng số 0 giúp giữ lại toàn bộ tín hiệu ngắn hơn một giây và đưa chúng về cùng độ dài. Truncating giúp giới hạn các tín hiệu dài hơn về đúng kích thước quy định. Sau khi MFCC được mean-pooling theo trục thời gian, mỗi mẫu đều trở thành vector 13 chiều cố định, phù hợp với đầu vào của SVM.

Nếu bỏ qua bước này, có thể xảy ra lỗi kích thước khi ghép dữ liệu hoặc mỗi mẫu phải bị biến đổi không nhất quán. Điều đó làm mất tính tương thích giữa các mẫu, khiến quá trình huấn luyện SVM không thể thực hiện hoặc tạo ra đặc trưng không công bằng giữa các file.

**Câu hỏi 2:** Thuật toán MFCCs được coi là rất phù hợp với các mô hình Machine Learning truyền thống (như SVM, GMM) so với Log-Mel Spectrogram. Hãy dựa vào kiến thức về phép biến đổi DCT (Discrete Cosine Transform) để giải thích tại sao?

*Trả lời của sinh viên:*

MFCC được tạo từ Log-Mel Spectrogram, sau đó áp dụng phép biến đổi DCT lên trục các kênh Mel. DCT biến đổi các giá trị năng lượng tương quan thành các hệ số cepstral ít tương quan hơn. Vì vậy, thông tin được tập trung vào một số hệ số đầu, thường giữ lại khoảng 12 đến 13 hệ số MFCC.

So với Log-Mel Spectrogram có hàng nghìn giá trị phụ thuộc vào số frame thời gian và số dải Mel, MFCC sau mean-pooling tạo ra vector đặc trưng nhỏ gọn, ổn định và có số chiều cố định. Điều này phù hợp với SVM hoặc GMM vì các mô hình truyền thống hoạt động tốt trên dữ liệu có số chiều vừa phải và ít tương quan.

DCT cũng giúp giảm dư thừa thông tin và làm giảm ảnh hưởng của các biến thiên chậm của phổ, chẳng hạn sự khác biệt về thiết bị ghi âm hoặc môi trường. Các hệ số MFCC đầu thường mô tả bao phổ của giọng nói, là thông tin quan trọng cho nhận dạng từ khóa. Ngược lại, Log-Mel Spectrogram giữ lại nhiều thông tin không gian-thời gian hơn và thường phù hợp hơn với CNN hoặc các mô hình Deep Learning có khả năng tự học biểu diễn.